# 03 · LightGBM triage baseline (Alert Triage #01)
Fit the classifier the pipeline fits, on the same split, and look at what it learned. The model class is `src/models/lightgbm_triage.LightGBMTriage`.

In [ ]:
import sys
from pathlib import Path
SLOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(SLOT))
from src.config import load_all_configs
from src import data_source as ds
cfg = load_all_configs()
mcfg, fcfg = cfg['model'], cfg['feature']
selected = ds.load_selected(mcfg)['features']
print('dataset', mcfg['data']['dataset'], '| model features', selected)


## Split and fit

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from src.models.lightgbm_triage import LightGBMTriage
df = ds.load_frame(mcfg, columns=selected)
y = df[mcfg['target']].astype(int).values
tr, te = train_test_split(np.arange(len(df)), test_size=mcfg['split']['test_frac'],
                          random_state=mcfg['seed'], stratify=y)
model = LightGBMTriage(mcfg['lgbm_classifier'], mcfg['seed']).fit(
            df.iloc[tr][selected], selected, y[tr])
print('scale_pos_weight (from the split, not configured):', model.scale_pos_weight)

## Score the held-out split

In [ ]:
p = model.attack_probability(df.iloc[te][selected])
from src.evaluation import attack_class_evaluator as ace
thr = model.fit_threshold(y[te], p, mcfg['operating_point']['min_precision'],
                          mcfg['operating_point']['fallback_threshold'])
print('fitted threshold:', round(thr, 6))
ace.evaluate_flows(y[te], p, model.verdict(p), model.attack_probability(df.iloc[tr][selected]), y[tr])

## Split-gain importance

In [ ]:
model.gain_importance().sort_values().plot.barh(figsize=(8,4), color='#2a78d6',
        title='% of total gain');

## Why a flow scored — exact tree SHAP from the booster itself

In [ ]:
import pandas as pd
Xs = df.iloc[te][selected].head(2000)
sv = model.shap_contributions(Xs)
ps = model.attack_probability(Xs); i = int(ps.argmax())
print('explaining the highest-scoring flow, p =', round(float(ps[i]), 6))
pd.Series(sv[i], index=selected).sort_values()